In [26]:
import os
import sys
import time
import requests
import rasterio
import geopandas as gpd
from osgeo import gdal
import math
import numpy as np
from rasterio.transform import from_origin
from pyproj import Transformer

# Dynamic dataset needed are ERA-5 variables, Thuenen crop-type maps, Sentinel-2 and Sentinel-3 dataset

origin = '/data/Aldhani/eoagritwin/'
sys.path.append('/home/potzschf/repos/')

from FieldWaterUseTools.FuncBox.Misc import getFilelist, path_safe, slash_checker, convertVRTpathsTOrelative, download_thuenen_cropTypes # 
from FieldWaterUseTools.FuncBox.DICT_LIST import REAL_INT_TO_MONTH # FieldWaterUseTools.FuncBox.

storPath_master = slash_checker(path_safe(f"{origin}et/Z_REPO_TEST/"))#'/place/to/store/porducts/')
path_to_grid = f"{storPath_master}GRID/Gridding_ET_FORCE.gpkg"
######################################################################### DEM


storPath_DEM = slash_checker(path_safe(f"{storPath_master}DEM/raw_tiles/"))
storPath_DEM_tiled = slash_checker(path_safe(f"{storPath_master}DEM/Gridded/"))

In [ ]:

# get metadata



In [24]:
transform

Affine(20.0, 0.0, 4016026.363042,
       0.0, -20.0, 3584919.607965)

In [27]:

grid_GER = gpd.read_file(path_to_grid)
grid_GER = grid_GER.to_crs("EPSG:3035") # it is already at this EPSG


In [ ]:
minx, miny, maxx, maxy = grid_GER.total_bounds

pixel_size = 20 # as gpd was created from raster at this resolution

n_pixels_x = (maxx - minx) / pixel_size
n_pixels_y = (maxy - miny) / pixel_size

width = int(math.ceil(n_pixels_x))
height = int(math.ceil(n_pixels_y))

# Raster transform
transform = from_origin(
    minx,
    maxy,
    pixel_size,
    pixel_size
)

# create grid
cols, rows = np.meshgrid(np.arange(width), np.arange(height))

# get center coordinates of pixel
xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')
xs = np.array(xs).reshape((height, width))
ys = np.array(ys).reshape((height, width))

In [ ]:
with rasterio.open('/data/Aldhani/eoagritwin/et/Auxiliary/DEM/vrt_and_derivates/DEM_GER.vrt') as src:
    width = src.width
    height = src.height
    transform = src.transform
    crs_src = src.crs 
    
# create grid
cols, rows = np.meshgrid(np.arange(width), np.arange(height))

# get center coordinates of pixel
xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')
xs = np.array(xs).reshape((height, width))
ys = np.array(ys).reshape((height, width))

# export
out_meta = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": grid_GER.crs,    
    "transform": transform,
    "nodata": -9999
}
with rasterio.open('/data/Aldhani/eoagritwin/et/Auxiliary/DEM/vrt_and_derivates/LON_GER.tif', 'w', **out_meta) as dst:
    dst.write(lons.astype('float32'), 1)

# Example to save lat raster
with rasterio.open('/data/Aldhani/eoagritwin/et/Auxiliary/DEM/vrt_and_derivates/LAT_GER.tif', 'w', **out_meta) as dst:
    dst.write(lats.astype('float32'), 1)

/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/


['/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_0_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_0_2.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_1_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_2_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_3_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_4_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_5_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_6_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_7_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_7_1.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_8_0.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_8_1.tif',
 '/data/Aldhani/eoagritwin/et/Z_REPO_TEST/DEM/raw_tiles/DEM_GER_9_1.tif']

In [ ]:
vrt_name = f'{storPath_DEM}DEM.vrt'
vrt = gdal.BuildVRT(vrt_name, getFilelist(storPath_DEM, ".tif"), separate = False)
vrt = None
#convertVRTpathsTOrelative(vrt_name)

/data/Aldhani/users/potzschf/conda/envs/workhorse/lib/python3.12/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [ ]:
# now cut DEM, aspect, slope into FORCE TILES
grid_GER = gpd.read_file(path_to_grid)
DEM_vrt = 
with rasterio.open(DEM_vrt) as src:

    for tileID in grid_GER['filename']:
        tile = grid_GER[grid_GER['filename'] == tileID]
        # print(Vector.crs)
        out_image, out_transform=rasterio.mask.mask(src,tile.geometry,crop=True)
        out_meta=src.meta.copy() # copy the metadata of the source DEM
        
        out_meta.update({
            "driver":"Gtiff",
            "height":out_image.shape[1], # height starts with shape[1]
            "width":out_image.shape[2], # width starts with shape[2]
            "transform":out_transform
        })
                
        with rasterio.open(f'{storPath_DEM_tiled}/DEM_{tileID}.tif','w',**out_meta) as dst:
            dst.write(out_image)

In [ ]:
# set year and month for which to obtain data

YEAR = 2020
MONTH = 5

# set masterpath for stored data

print(storPath_master)
storPath_S2 = path_safe(f"{storPath_master}{YEAR}/{MONTH:02d}/S2/")
storPath_S3 = path_safe(f"{storPath_master}{YEAR}/{MONTH:02d}/S3/")
storPath_ERA5 = path_safe(f"{storPath_master}{YEAR}/{MONTH:02d}/ERA5/")
path_to_geopot_raw = f"{storPath_ERA5}GEOPOT/"
path_to_Thuenen = path_safe(f"{storPath_master}{YEAR}/Thuenen/")

/data/Aldhani/eoagritwin/Z_REPO_TEST/


In [12]:
storPath_master = slash_checker(path_safe(f"{origin}Z_REPO_TEST"))#'/place/to/store/porducts/'))
storPath_S2 = path_safe(f"{storPath_master}{YEAR}/{MONTH:02d}/S2")
storPath_S3 = path_safe(f"{storPath_master}{YEAR}/{MONTH:02d}/S3")
storPath_ERA5 = path_safe(f"{storPath_master}{YEAR}/{MONTH:02d}/ERA5/")
storPath_GeoPot = f"{storPath_ERA5}GEOPOT/"
storPath_Thuen = path_safe(f"{storPath_master}{YEAR}/Thuenen/")

